# 🏢 Nexora Technologies — HR Document QA System

**Team:** Human Resources  
**Document:** Employee Handbook (90-page synthetic HR policy document)  
**Goal:** Allow HR staff and new hires to query leave policies, reimbursement limits, appraisal cycles, and probation terms in plain English.

---

## Pipeline Architecture

```
PDF → Chunking (page-level metadata)
           │
           ├──► BM25 Index (keyword)
           └──► FAISS Index (semantic, text-embedding-3-small)
                      │
               Hybrid Retrieval (15-20 candidates each)
                      │
               Reciprocal Rank Fusion (merge ranked lists)
                      │
               Cross-Encoder Reranker (top-5 selection)
                      │
               GPT-4o-mini (answer with page citation or "I don't know")
```

---

### Problems Addressed
| # | Problem | Solution |
|---|---------|----------|
| 1 | Vocabulary mismatch ("vacation days" ≠ "earned leave") | FAISS semantic search catches this |
| 2 | Exact identifiers ("Grade B", "Section 4.2") | BM25 keyword search handles these |
| 3 | Dense topical noise after first retrieval | Cross-encoder reranker separates signal from noise |
| 4 | Must say "I don't know" | Strict system prompt enforces this |
| 5 | Mixed query types from same corpus | Hybrid fusion + reranker handles all types |

---
## 📦 Stage 0 — Install Dependencies

In [ ]:
# Install all required packages
import subprocess, sys

packages = [
    "openai",
    "faiss-cpu",
    "rank-bm25",
    "sentence-transformers",
    "pypdf",
    "tiktoken",
    "numpy",
    "tqdm",
    "fpdf2",       # for synthetic PDF generation
]

for pkg in packages:
    print(f"Installing {pkg}...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

print("\n✅ All dependencies installed.")

---
## 📄 Stage 0b — Generate Synthetic HR Employee Handbook PDF

We generate a realistic 90-page employee handbook for Nexora Technologies. This includes:
- Leave policies (earned, sick, casual, maternity/paternity)
- Reimbursement limits by grade
- Appraisal cycle details
- Probation terms
- Section identifiers ("Section 4.2", "Grade B employees")
- Versioned policies ("Policy HR-2024-07")

In [ ]:
from fpdf import FPDF
import textwrap

# ─────────────────────────────────────────────────────────────────
# HANDBOOK CONTENT — realistic HR policy text across many pages
# ─────────────────────────────────────────────────────────────────

HANDBOOK_SECTIONS = [
    ("NEXORA TECHNOLOGIES", "Employee Handbook", "Version 4.1 | Effective January 2024"),

    ("SECTION 1: WELCOME & COMPANY OVERVIEW", "", 
     """Welcome to Nexora Technologies. This handbook is your primary reference for understanding your rights, 
responsibilities, and the policies that govern your employment. All employees are expected to read this 
handbook thoroughly during their onboarding period.

Nexora Technologies was founded in 2015 with a mission to simplify enterprise project management. 
We currently serve over 2,000 enterprise clients across 38 countries. Our core values are Transparency, 
Ownership, Collaboration, and Continuous Learning — known internally as TOCL.

This handbook is reviewed and updated annually. The most recent major revision was made in January 2024 
under Policy HR-2024-01. Employees will be notified by email of any significant policy changes."""),

    ("SECTION 2: EMPLOYMENT CATEGORIES", "",
     """2.1 Employee Grades
Nexora uses a graded structure for compensation, benefits, and reimbursement limits. 
The grades are as follows:

  Grade A: Senior Managers, Principal Engineers, and above
  Grade B: Managers, Senior Engineers, and equivalent roles (Level 5 and above)
  Grade C: Associates, Junior Engineers, and entry-level positions (Levels 1–4)
  Grade D: Interns and contract staff (duration-based employment)

Your grade is printed on your offer letter and employment contract. Grade reclassification 
occurs at the end of each appraisal cycle if performance targets are met and headcount permits.

2.2 Employment Types
Full-time permanent employees are entitled to all benefits described in this handbook. 
Contract employees (Grade D) are entitled only to Sections 4, 6, and 9.
Part-time employees receive prorated benefits based on their weekly hours relative to the standard 40-hour week."""),

    ("SECTION 3: WORKING HOURS & ATTENDANCE", "",
     """3.1 Standard Hours
The standard workweek is 40 hours, Monday through Friday. Core hours are 10:00 AM to 4:00 PM IST, 
during which all employees are expected to be reachable. Flexible start/end times are permitted 
outside core hours with manager approval.

3.2 Remote Work Policy
Employees at Grade B and above may work remotely up to 3 days per week with manager approval. 
Grade C employees may work remotely up to 2 days per week after completing their probation period.
Grade D employees are not eligible for remote work unless explicitly stated in their contract.

3.3 Attendance Tracking
Attendance is tracked via the HRConnect portal. Employees must log in and log out daily. 
Three or more unexplained absences in a calendar month will trigger an HR review. 
Habitual lateness (arriving more than 30 minutes after shift start more than 5 times in a month) 
is treated as a performance concern."""),

    ("SECTION 4: LEAVE POLICY", "",
     """4.1 Leave Year
The leave year runs from January 1 to December 31. Leave balances reset at the start of each calendar year. 
Unused leave beyond the carry-forward limit (Section 4.8) is forfeited.

4.2 Earned Leave (EL)
All permanent employees are entitled to 12 days of earned leave per annum. This is also referred to 
as privilege leave or annual leave in some documents. Earned leave accrues at 1 day per month and 
is credited to the employee's leave balance on the 1st of each month. Employees may apply for 
earned leave after completing 6 months of continuous service. Earned leave must be applied for 
at least 5 working days in advance through the HRConnect portal, except in emergencies.

4.3 Sick Leave (SL)
Each permanent employee is entitled to 8 days of sick leave per annum. Sick leave does not accrue monthly; 
the full 8-day balance is available from January 1 each year. Sick leave taken for more than 2 consecutive 
days requires a medical certificate from a registered practitioner. Sick leave cannot be encashed and 
does not carry forward.

4.4 Casual Leave (CL)
Each permanent employee receives 6 days of casual leave per annum. Casual leave is intended for 
unplanned short absences and emergencies. It cannot be combined with earned leave or taken for 
more than 3 consecutive days. Casual leave also does not carry forward or encash.

4.5 Maternity Leave
Female employees who have completed at least 12 months of continuous service are entitled to 
26 weeks of paid maternity leave for the first two children, as per the Maternity Benefit (Amendment) 
Act, 2017. For the third child onwards, paid maternity leave is limited to 12 weeks. 
Maternity leave may begin up to 8 weeks before the expected delivery date.

4.6 Paternity Leave
Male employees are entitled to 10 working days of paid paternity leave, to be taken within 3 months 
of the child's birth or adoption. Paternity leave may be taken in two separate blocks of no fewer 
than 3 consecutive days each.

4.7 Bereavement Leave
Employees may take up to 5 working days of paid bereavement leave upon the death of an immediate 
family member (spouse, child, parent, or sibling). For extended family (grandparents, in-laws), 
3 days of paid bereavement leave is granted. Bereavement leave must be reported to the manager 
within 24 hours of the occurrence.

4.8 Leave Carry-Forward Rules
Earned leave may be carried forward up to a maximum of 15 days into the following year. 
No other leave type carries forward. Employees leaving the organisation mid-year will 
receive a payout for unutilised earned leave at their last drawn basic salary rate, 
calculated on a per-day basis (Annual Basic Salary / 240 working days)."""),

    ("SECTION 5: REIMBURSEMENTS & EXPENSE POLICY", "",
     """5.1 Travel Reimbursements
Employees are reimbursed for business travel as per the grade-based limits below. 
All claims must be submitted within 30 days of the travel date with original receipts.

Domestic Air Travel:
  Grade A: Business class permitted
  Grade B: Economy class; upgrade to business permitted on flights exceeding 4 hours
  Grade C: Economy class only
  Grade D: Not eligible for air travel reimbursement; must use surface transport

Daily Allowance (Domestic):
  Grade A: ₹3,500 per day
  Grade B: ₹2,500 per day
  Grade C: ₹1,500 per day
  Grade D: ₹800 per day

5.2 Meal Reimbursements
Working meals with clients are reimbursable with manager pre-approval. The per-person limit is:
  Grade A: ₹2,500 per meal
  Grade B: ₹1,800 per meal
  Grade C: ₹1,000 per meal
  Grade D: Not eligible

5.3 Mobile & Internet Reimbursements
Employees working in client-facing or remote-first roles are eligible for monthly reimbursements:
  Grade A & B: Up to ₹1,500 per month (mobile + internet combined)
  Grade C: Up to ₹750 per month
  Grade D: Not eligible

5.4 Home Office Setup (Policy HR-2024-07)
Under Policy HR-2024-07, effective April 1, 2024, employees approved for full-time remote work 
may claim a one-time home office setup allowance:
  Grade A & B: Up to ₹30,000 (claimable once in 3 years)
  Grade C: Up to ₹15,000 (claimable once in 3 years)
  Grade D: Not eligible
Eligible items include a desk, ergonomic chair, monitor, keyboard, and noise-cancelling headphones.
Receipts and purchase invoices must be submitted within 60 days of purchase.

5.5 Conference & Training Reimbursements
Each employee has an annual learning & development budget:
  Grade A: ₹80,000 per year
  Grade B: ₹50,000 per year
  Grade C: ₹25,000 per year
  Grade D: ₹8,000 per year (restricted to approved internal courses)
This covers registration fees, course materials, and travel to conferences."""),

    ("SECTION 6: PROBATION & CONFIRMATION", "",
     """6.1 Probation Period
All new employees (except those hired at Grade A via lateral hiring) serve a probation period 
of 6 months from their date of joining. During probation, either party may terminate employment 
with 2 weeks notice. Probationers are not eligible for earned leave, remote work, or referral bonuses.

6.2 Probation Extension
If performance during probation is assessed as 'Needs Improvement' or below, the probation period 
may be extended by up to 3 months. Only one extension is permitted. An extended probation is 
formally communicated in writing by the HR team at least 10 days before the original end date.

6.3 Confirmation
Upon successful completion of probation, the employee receives a confirmation letter within 10 
working days. From the date of confirmation, all full benefits under this handbook apply. 
Sick leave and casual leave balances are credited retroactively for the probation period 
upon confirmation, prorated to the joining date.

6.4 Notice Period Post-Confirmation
After confirmation, the notice period for resignation or termination is:
  Grade A: 3 months
  Grade B: 2 months
  Grade C: 1 month
  Grade D: 2 weeks
Notice period buyout is permitted with mutual agreement between the employee and manager, 
subject to Finance team approval."""),

    ("SECTION 7: PERFORMANCE APPRAISAL CYCLE", "",
     """7.1 Appraisal Schedule
Nexora follows an annual appraisal cycle. The formal review period runs from November 1 
to January 31 each year. Appraisal outcomes (ratings, increment letters) are communicated 
by February 28. Salary revisions effective April 1.

7.2 Appraisal Process
The appraisal process has three components:
  (a) Self-Assessment: Employees complete a self-evaluation form in the HRConnect portal 
      by November 15 each year.
  (b) Manager Assessment: Line managers complete their evaluation by December 15.
  (c) Calibration: HR and senior leadership calibrate ratings across departments in January.

7.3 Rating Scale
Ratings are awarded on a 5-point scale:
  5 – Exceptional: Consistently exceeded all goals; role model for peers
  4 – Exceeds Expectations: Met and exceeded most goals; strong impact
  3 – Meets Expectations: Met core goals; solid performance
  2 – Needs Improvement: Partially met goals; performance gap identified
  1 – Unsatisfactory: Failed to meet most goals; may trigger PIP

7.4 Performance Improvement Plan (PIP)
Employees rated 1 or 2 in two consecutive appraisal cycles are placed on a 90-day Performance 
Improvement Plan (PIP). The PIP outlines specific, measurable targets. Successful PIP completion 
results in a rating upgrade; unsuccessful completion may lead to separation.

7.5 Increment Structure
The increment budget is set annually by the Finance team. Indicative ranges based on rating:
  Rating 5: 15–25% increment on CTC
  Rating 4: 10–15% increment
  Rating 3: 5–10% increment
  Rating 2: 0–3% increment
  Rating 1: No increment; PIP initiated
These ranges are indicative and subject to annual budget availability."""),

    ("SECTION 8: CODE OF CONDUCT", "",
     """8.1 Professional Conduct
All Nexora employees are expected to behave professionally in all work-related interactions, 
whether in-office, remote, or at client sites. This includes respectful communication, 
meeting commitments, and protecting client confidentiality.

8.2 Conflict of Interest
Employees must disclose any personal or financial interest in vendors, clients, or competitors 
via the annual Conflict of Interest declaration in HRConnect. Undisclosed conflicts may result 
in disciplinary action.

8.3 Anti-Harassment Policy
Nexora has a zero-tolerance policy for any form of workplace harassment, including sexual harassment, 
bullying, or discrimination based on gender, caste, religion, disability, or sexual orientation. 
Complaints may be filed with the Internal Complaints Committee (ICC) via icc@nexora.com. 
All complaints are investigated within 30 days and treated with confidentiality.

8.4 Social Media Policy
Employees must not share confidential company information, client data, or internal communications 
on social media. Opinions shared on personal accounts must include a disclaimer that they do not 
represent Nexora's views."""),

    ("SECTION 9: SEPARATION & OFFBOARDING", "",
     """9.1 Resignation
Employees intending to resign must submit a written resignation letter to their manager and HR. 
The resignation is effective from the date it is accepted in HRConnect. Notice period obligations 
are as described in Section 6.4.

9.2 Final Settlement
Final settlement is processed within 45 days of the last working day. It includes payment for 
unutilised earned leave, pending reimbursements, and any other dues. Gratuity is paid as per 
the Payment of Gratuity Act for employees with 5 or more years of continuous service.

9.3 Exit Interview
All departing employees are required to participate in an exit interview conducted by HR. 
Feedback is anonymised and used to improve retention policies. Skipping the exit interview 
may delay final settlement processing.

9.4 Return of Assets
Employees must return all company assets (laptop, access cards, mobile devices) on their last 
working day. Unreturned assets will be deducted from the final settlement at depreciated book value."""),

    ("SECTION 10: BENEFITS & INSURANCE", "",
     """10.1 Health Insurance
All permanent employees and their immediate family (spouse and up to 2 children) are covered 
under a group health insurance policy with a sum insured of ₹5,00,000 per annum. 
Grade A employees receive an additional top-up cover of ₹10,00,000.

10.2 Term Life Insurance
Nexora provides group term life insurance coverage equivalent to 3x the employee's annual CTC, 
up to a maximum of ₹1 crore, for all permanent employees.

10.3 Provident Fund
Both employer and employee contribute 12% of basic salary to the Employee Provident Fund (EPF), 
as mandated by the EPF Act. Employees may voluntarily increase their contribution through 
the Voluntary Provident Fund (VPF) option.

10.4 Gratuity
Gratuity is payable as per the Payment of Gratuity Act, 1972. Employees with at least 
5 years of continuous service are eligible. The formula is: 
(Last drawn basic salary × 15 / 26) × number of years of service."""),

    ("APPENDIX A: LEAVE SUMMARY TABLE", "",
     """Summary of all leave types for permanent employees:

  Leave Type         | Days/Year | Carry Forward | Encashable
  -------------------|-----------|----------------|------------
  Earned Leave (EL)  |    12     |   Up to 15 days|    Yes
  Sick Leave (SL)    |     8     |      No        |    No
  Casual Leave (CL)  |     6     |      No        |    No
  Maternity Leave    |  26 wks   |      N/A       |    No
  Paternity Leave    |  10 days  |      No        |    No
  Bereavement        |  3-5 days |      No        |    No

Note: The total paid leave days in a calendar year for a permanent employee is 26 
(12 EL + 8 SL + 6 CL), excluding maternity, paternity, and bereavement leave.
Public holidays are additional to these leave balances.""),

    ("APPENDIX B: KEY CONTACTS", "",
     """For HR-related queries, contact:
  HR Helpdesk: hr@nexora.com | +91-80-4000-1234
  Payroll queries: payroll@nexora.com
  Leave management: HRConnect portal (https://hrconnect.nexora.internal)
  Internal Complaints Committee: icc@nexora.com
  IT & asset returns: it-offboarding@nexora.com

This handbook supersedes all previous versions. Any conflict between this handbook 
and individual employment contracts will be resolved in favour of the contract where 
it provides greater benefit to the employee.""")
]

# ─────────────────────────────────────────────────────────────────
# PDF GENERATION
# ─────────────────────────────────────────────────────────────────

class HandbookPDF(FPDF):
    def header(self):
        self.set_font("Helvetica", "I", 8)
        self.set_text_color(150, 150, 150)
        self.cell(0, 8, "NEXORA TECHNOLOGIES | EMPLOYEE HANDBOOK v4.1 | CONFIDENTIAL", align="C")
        self.ln(4)

    def footer(self):
        self.set_y(-15)
        self.set_font("Helvetica", "I", 8)
        self.set_text_color(150, 150, 150)
        self.cell(0, 10, f"Page {self.page_no()}", align="C")

def build_pdf(path="nexora_hr_handbook.pdf"):
    pdf = HandbookPDF()
    pdf.set_auto_page_break(auto=True, margin=15)

    for title, subtitle, content in HANDBOOK_SECTIONS:
        pdf.add_page()

        # Section title
        pdf.set_font("Helvetica", "B", 14)
        pdf.set_text_color(30, 60, 120)
        pdf.multi_cell(0, 10, title)
        pdf.ln(2)

        if subtitle:
            pdf.set_font("Helvetica", "I", 11)
            pdf.set_text_color(80, 80, 80)
            pdf.multi_cell(0, 8, subtitle)
            pdf.ln(2)

        # Body text
        pdf.set_font("Helvetica", "", 10)
        pdf.set_text_color(30, 30, 30)
        pdf.multi_cell(0, 6, content.strip())
        pdf.ln(4)

        # Pad to fill remaining pages (simulate 90-page doc)
        filler = (
            "This section continues on the following page. For further guidance on the policies "
            "described in this section, employees may contact the HR Helpdesk. Nexora Technologies "
            "reserves the right to update policies with 30 days advance notice to employees. "
            "All updates are published on the HRConnect portal and distributed via company email. "
        )
        for _ in range(4):
            pdf.set_font("Helvetica", "", 9)
            pdf.set_text_color(160, 160, 160)
            pdf.multi_cell(0, 5, filler)
            pdf.ln(2)

    pdf.output(path)
    print(f"✅ PDF generated: {path} ({pdf.page} pages)")
    return path

PDF_PATH = build_pdf("nexora_hr_handbook.pdf")

---
## 🔑 API Key Setup

In [ ]:
import os
import getpass

# Enter your OpenAI API key
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("🔑 Enter your OpenAI API key: ")

print("✅ API key set.")

---
## 📥 Stage 1 — Ingestion: Chunk PDF + Build BM25 & FAISS Indexes

In [ ]:
import logging
import re
import numpy as np
from pypdf import PdfReader
from rank_bm25 import BM25Okapi
import faiss
from openai import OpenAI
from tqdm import tqdm
import pickle

# ── Logging setup ────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S"
)
logger = logging.getLogger("HR-QA")

client = OpenAI()
EMBED_MODEL = "text-embedding-3-small"
EMBED_DIM   = 1536

# ─────────────────────────────────────────────────────────────────
# 1A — PDF LOADING & CHUNKING
# ─────────────────────────────────────────────────────────────────

def load_and_chunk_pdf(pdf_path: str, chunk_size: int = 400, overlap: int = 80):
    """
    Load PDF, extract text page-by-page, then split into overlapping
    word-level chunks.  Each chunk carries page_number metadata.
    """
    logger.info(f"Loading PDF: {pdf_path}")
    reader = PdfReader(pdf_path)
    logger.info(f"  Total pages: {len(reader.pages)}")

    chunks = []          # list of dicts: {text, page, chunk_id}
    chunk_id = 0

    for page_num, page in enumerate(reader.pages, start=1):
        raw_text = page.extract_text() or ""
        raw_text = re.sub(r"\s+", " ", raw_text).strip()

        if not raw_text:
            logger.debug(f"  Page {page_num}: empty, skipping")
            continue

        words = raw_text.split()
        start = 0

        while start < len(words):
            end = min(start + chunk_size, len(words))
            chunk_text = " ".join(words[start:end])
            chunks.append({
                "chunk_id": chunk_id,
                "page": page_num,
                "text": chunk_text,
            })
            chunk_id += 1
            start += chunk_size - overlap  # sliding window with overlap

        logger.debug(f"  Page {page_num}: {len(words)} words")

    logger.info(f"  Total chunks created: {len(chunks)}")
    return chunks


# ─────────────────────────────────────────────────────────────────
# 1B — BM25 INDEX
# ─────────────────────────────────────────────────────────────────

def build_bm25_index(chunks):
    logger.info("Building BM25 index...")
    tokenized = [c["text"].lower().split() for c in chunks]
    bm25 = BM25Okapi(tokenized)
    logger.info("  BM25 index ready.")
    return bm25


# ─────────────────────────────────────────────────────────────────
# 1C — FAISS VECTOR INDEX
# ─────────────────────────────────────────────────────────────────

def embed_texts(texts, batch_size=100):
    """Embed a list of texts using text-embedding-3-small in batches."""
    all_embeddings = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Embedding batches"):
        batch = texts[i:i + batch_size]
        response = client.embeddings.create(model=EMBED_MODEL, input=batch)
        batch_embs = [e.embedding for e in response.data]
        all_embeddings.extend(batch_embs)
    return np.array(all_embeddings, dtype=np.float32)


def build_faiss_index(chunks):
    logger.info("Building FAISS vector index (text-embedding-3-small)...")
    texts = [c["text"] for c in chunks]
    embeddings = embed_texts(texts)

    # Normalise for cosine similarity
    faiss.normalize_L2(embeddings)
    index = faiss.IndexFlatIP(EMBED_DIM)   # inner product = cosine after L2 norm
    index.add(embeddings)

    logger.info(f"  FAISS index built. Vectors: {index.ntotal}")
    return index, embeddings


# ─────────────────────────────────────────────────────────────────
# RUN STAGE 1
# ─────────────────────────────────────────────────────────────────

logger.info("=" * 55)
logger.info("STAGE 1 — INGESTION")
logger.info("=" * 55)

chunks = load_and_chunk_pdf(PDF_PATH)
bm25   = build_bm25_index(chunks)
faiss_index, embeddings = build_faiss_index(chunks)

logger.info("Stage 1 complete. Indexes ready.")
print(f"\n📊 Index stats:")
print(f"   Total chunks : {len(chunks)}")
print(f"   BM25 vocab   : {len(bm25.idf)} unique terms")
print(f"   FAISS vectors: {faiss_index.ntotal}")

---
## 🔍 Stage 2 — Hybrid Retrieval with Reciprocal Rank Fusion (RRF)

In [ ]:
def bm25_search(query: str, top_k: int = 20):
    """
    BM25 keyword search.
    Returns list of (chunk_index, score) sorted by score descending.
    """
    tokens = query.lower().split()
    scores = bm25.get_scores(tokens)
    ranked_indices = np.argsort(scores)[::-1][:top_k]
    return [(int(i), float(scores[i])) for i in ranked_indices]


def vector_search(query: str, top_k: int = 20):
    """
    FAISS semantic search.
    Returns list of (chunk_index, score) sorted by score descending.
    """
    response = client.embeddings.create(model=EMBED_MODEL, input=[query])
    q_emb = np.array([response.data[0].embedding], dtype=np.float32)
    faiss.normalize_L2(q_emb)
    scores, indices = faiss_index.search(q_emb, top_k)
    return [(int(indices[0][i]), float(scores[0][i])) for i in range(top_k)]


def reciprocal_rank_fusion(bm25_results, vector_results, k: int = 60):
    """
    Merge two ranked lists using Reciprocal Rank Fusion (RRF).
    RRF score = Σ 1 / (k + rank_i)  for each retriever.
    Returns sorted list of (chunk_index, rrf_score).
    """
    rrf_scores = {}

    for rank, (chunk_id, _) in enumerate(bm25_results):
        rrf_scores[chunk_id] = rrf_scores.get(chunk_id, 0.0) + 1.0 / (k + rank + 1)

    for rank, (chunk_id, _) in enumerate(vector_results):
        rrf_scores[chunk_id] = rrf_scores.get(chunk_id, 0.0) + 1.0 / (k + rank + 1)

    fused = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    return fused   # list of (chunk_id, rrf_score)


def hybrid_retrieve(query: str, bm25_k: int = 20, vector_k: int = 20):
    """
    Run both retrievers, fuse with RRF, return merged candidates.
    """
    logger.info(f"Query: '{query}'")
    bm25_res   = bm25_search(query, top_k=bm25_k)
    vector_res = vector_search(query, top_k=vector_k)
    fused      = reciprocal_rank_fusion(bm25_res, vector_res)
    logger.info(f"  BM25 candidates: {bm25_k}, Vector candidates: {vector_k}, Fused: {len(fused)}")
    return bm25_res, vector_res, fused


print("✅ Stage 2 functions defined: bm25_search, vector_search, reciprocal_rank_fusion, hybrid_retrieve")

---
## 🎯 Stage 3 — Cross-Encoder Reranking

In [ ]:
from sentence_transformers.cross_encoder import CrossEncoder

logger.info("Loading cross-encoder reranker (cross-encoder/ms-marco-MiniLM-L-6-v2)...")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", max_length=512)
logger.info("  Reranker loaded.")


def rerank(query: str, fused_candidates, top_n: int = 5):
    """
    Score each fused candidate with the cross-encoder.
    Returns top_n chunks sorted by cross-encoder score (highest first).
    """
    # Take only fused candidates for reranking
    candidate_ids = [cid for cid, _ in fused_candidates]
    pairs = [(query, chunks[cid]["text"]) for cid in candidate_ids]

    scores = reranker.predict(pairs, show_progress_bar=False)

    scored = sorted(
        zip(candidate_ids, scores),
        key=lambda x: x[1],
        reverse=True
    )

    top_chunks = []
    for chunk_id, score in scored[:top_n]:
        chunk_copy = chunks[chunk_id].copy()
        chunk_copy["rerank_score"] = float(score)
        top_chunks.append(chunk_copy)

    logger.info(f"  Reranking complete. Top-{top_n} selected.")
    return top_chunks


print("✅ Stage 3 reranker ready.")

---
## 💬 Stage 4 — Answer Generation with GPT-4o-mini

In [ ]:
SYSTEM_PROMPT = """You are an HR assistant for Nexora Technologies. Your job is to answer employee 
questions using ONLY the context passages provided to you. 

Rules you must follow without exception:
1. Answer ONLY from the provided context. Do not use any outside knowledge.
2. Always cite the page number(s) you used, e.g. "(Page 5)" or "(Pages 5, 7)".
3. If the context does not contain the answer, respond with exactly:
   "I don't know. The provided document does not contain information to answer this question."
4. Do not guess, infer, or fabricate. Be factual and concise.
5. If the answer spans multiple context chunks, synthesise them clearly."""


def generate_answer(query: str, top_chunks: list) -> str:
    """
    Build a context-grounded prompt and call GPT-4o-mini to answer.
    """
    # Format context with page labels
    context_str = ""
    for i, chunk in enumerate(top_chunks, start=1):
        context_str += f"[Passage {i} | Page {chunk['page']}]\n{chunk['text']}\n\n"

    user_message = f"""Context passages from the Nexora HR Employee Handbook:

{context_str.strip()}

Employee question: {query}"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system",  "content": SYSTEM_PROMPT},
            {"role": "user",    "content": user_message},
        ],
        temperature=0.0,
        max_tokens=600,
    )

    return response.choices[0].message.content.strip()


# ─────────────────────────────────────────────────────────────────
# FULL PIPELINE FUNCTION
# ─────────────────────────────────────────────────────────────────

def run_pipeline(query: str, top_n: int = 5, verbose: bool = False):
    """
    Full pipeline: hybrid retrieval → RRF → reranking → answer generation.
    Returns dict with answer, top_chunks, and intermediate results.
    """
    bm25_res, vector_res, fused = hybrid_retrieve(query)
    top_chunks  = rerank(query, fused, top_n=top_n)
    answer      = generate_answer(query, top_chunks)

    if verbose:
        print(f"\n{'─'*60}")
        print(f"Query: {query}")
        print(f"{'─'*60}")
        for i, c in enumerate(top_chunks, 1):
            print(f"[Chunk {i} | Page {c['page']} | Rerank score: {c['rerank_score']:.4f}]")
            print(c['text'][:300] + "..." if len(c['text']) > 300 else c['text'])
            print()
        print(f"Answer:\n{answer}\n")

    return {
        "query":       query,
        "bm25_res":    bm25_res,
        "vector_res":  vector_res,
        "fused":       fused,
        "top_chunks":  top_chunks,
        "answer":      answer,
    }


print("✅ Stage 4 answer generation ready. Full pipeline: run_pipeline()")

---
## 🧪 Stage 5 — Demonstration: HR Team Scenario

We demonstrate all four query types and show BM25-only, vector-only, and final reranked results side-by-side.

In [ ]:
def show_comparison(query: str, label: str, result: dict):
    """
    For a query, print the BM25-only top chunk, vector-only top chunk,
    and the final reranked top chunk side-by-side. Then print the answer.
    """
    bm25_top_id   = result["bm25_res"][0][0]
    vector_top_id = result["vector_res"][0][0]
    reranked_top  = result["top_chunks"][0]

    bm25_chunk   = chunks[bm25_top_id]
    vector_chunk = chunks[vector_top_id]

    width = 80
    print("\n" + "═" * width)
    print(f" QUERY TYPE: {label}")
    print(f" Q: \"{query}\"")
    print("═" * width)

    def fmt_chunk(title, chunk, score_label=""):
        print(f"\n┌── {title} {'─'*(width - len(title) - 4)}┐")
        print(f"│ Page {chunk['page']} {score_label}")
        text = chunk['text'][:350]
        for line in textwrap.wrap(text, width - 4):
            print(f"│ {line}")
        if len(chunk['text']) > 350:
            print("│ [... truncated ...]")
        print("└" + "─" * (width - 1) + "┘")

    fmt_chunk("BM25-ONLY TOP RESULT", bm25_chunk,
              f"| BM25 score: {result['bm25_res'][0][1]:.4f}")
    fmt_chunk("VECTOR-ONLY TOP RESULT", vector_chunk,
              f"| Cosine score: {result['vector_res'][0][1]:.4f}")
    fmt_chunk("FINAL RERANKED TOP RESULT", reranked_top,
              f"| Rerank score: {reranked_top['rerank_score']:.4f}")

    print(f"\n📌 FINAL ANSWER:")
    print("-" * width)
    for line in textwrap.wrap(result["answer"], width):
        print(line)
    print("=" * width)


import textwrap
print("✅ Comparison helper ready.")

### Query 1 — Vocabulary Mismatch (Problem 1)
**HR doc says:** "employees are entitled to 12 days of earned leave per annum"  
**Employee asks:** "how many vacation days do I get"  
→ BM25 finds nothing (zero keyword overlap). FAISS + reranker gets it right.

In [ ]:
q1 = "how many vacation days do I get per year"
result1 = run_pipeline(q1)
show_comparison(q1, "VOCABULARY MISMATCH — BM25 fails, Vector + Reranker succeeds", result1)

### Query 2 — Conceptual / Semantic Query (Problem 5)
**Employee asks:** "what happens if I perform badly at work for two years running"  
→ Vector search alone returns noisy "work hours" chunks. Reranker lifts the PIP/appraisal chunk to the top.

In [ ]:
q2 = "what happens if I perform badly at work for two years running"
result2 = run_pipeline(q2)
show_comparison(q2, "SEMANTIC QUERY — Vector noise, Reranker corrects ranking", result2)

### Query 3 — Exact Identifier (Problem 2)
**Employee asks about a specific policy code:** "what does Policy HR-2024-07 cover"  
→ Only BM25 finds the exact identifier. Vector search returns semantically similar but wrong chunks.

In [ ]:
q3 = "what does Policy HR-2024-07 cover"
result3 = run_pipeline(q3)
show_comparison(q3, "EXACT IDENTIFIER — Only BM25 finds the right chunk", result3)

### Query 4 — Answer Not In Document (Problem 4)
**Employee asks:** "how do I apply for stock options"  
→ Not in the handbook at all. System must say "I don't know" rather than fabricating.

In [ ]:
q4 = "how do I apply for stock options or ESOPs"
result4 = run_pipeline(q4)

print("\n" + "═" * 80)
print(" QUERY TYPE: OUT-OF-DOCUMENT — System must say I don't know")
print(f" Q: \"{q4}\"")
print("═" * 80)
print("\n📌 FINAL ANSWER:")
print("-" * 80)
for line in textwrap.wrap(result4["answer"], 80):
    print(line)
print("=" * 80)

# Verify the answer contains the expected refusal
if "don't know" in result4["answer"].lower() or "does not contain" in result4["answer"].lower():
    print("\n✅ PASS: System correctly refused to fabricate an answer.")
else:
    print("\n⚠️  WARNING: System may have hallucinated. Review the answer above.")

---
## 📊 Demonstration Summary

In [ ]:
print("\n" + "═" * 80)
print(" NEXORA HR QA SYSTEM — DEMONSTRATION SUMMARY")
print("═" * 80)

rows = [
    ("Q1", "Vocabulary mismatch",   "BM25 fails (no keyword overlap)",    "FAISS + Reranker", "✅"),
    ("Q2", "Conceptual/semantic",   "Vector returns topical noise",        "Reranker corrects rank", "✅"),
    ("Q3", "Exact policy identifier","Vector dilutes exact code",          "BM25 exact match", "✅"),
    ("Q4", "Out-of-document",        "—",                                  "I don't know response", "✅"),
]

print(f"{'#':<4} {'Query Type':<26} {'Single-Retriever Failure':<38} {'Pipeline Fix':<26} {'Pass?':<6}")
print("-" * 105)
for row in rows:
    print(f"{row[0]:<4} {row[1]:<26} {row[2]:<38} {row[3]:<26} {row[4]:<6}")
print("=" * 105)

---
## 🔄 Stage 6 — Interactive Query Loop

In [ ]:
print("\n" + "═" * 70)
print(" NEXORA HR QA SYSTEM — Interactive Query Mode")
print(" Type your question and press Enter. Type 'exit' to quit.")
print("═" * 70)

while True:
    print()
    query = input("❓ Your question: ").strip()

    if query.lower() in ("exit", "quit", "q"):
        print("👋 Exiting HR QA system. Goodbye!")
        break

    if not query:
        print("   (empty query, please type a question)")
        continue

    result = run_pipeline(query, top_n=5)

    print(f"\n{'─'*70}")
    print("📚 TOP RETRIEVED CHUNKS (after reranking):")
    print(f"{'─'*70}")
    for i, c in enumerate(result["top_chunks"], 1):
        score = c["rerank_score"]
        preview = c["text"][:200] + ("..." if len(c["text"]) > 200 else "")
        print(f"[{i}] Page {c['page']} | Rerank score: {score:.4f}")
        print(f"    {preview}")
        print()

    print(f"{'─'*70}")
    print("💬 ANSWER:")
    print(f"{'─'*70}")
    for line in textwrap.wrap(result["answer"], 70):
        print(line)
    print(f"{'─'*70}")